# 1. Purpose and scope

This notebook executes Phase 5 validation-bootstrap stability analysis for the frozen ten-seed MentalRoBERTa experiment. It uses only the row-level validation predictions already stored in the ten artifact ZIPs.

It runs on CPU and does not load model weights, retrain, perform inference, use raw text, require a network connection, or access the locked test set. Phase 3 and Phase 4 remain frozen; this notebook creates a separate Phase 5 result package and freezes the final hard-pair set `H`.


# 2. User configuration

Place this notebook beside the directories shown below. These paths are the only settings a user normally needs to edit.


In [1]:
# USER CONFIGURATION
from pathlib import Path

INPUT_DIR = Path("mental_roberta_10_seeds results")
PHASE4_RESULTS_DIR = Path("phase4_boundary_analysis_results")
PHASE4_PAIR_METRICS_PATH = PHASE4_RESULTS_DIR / "phase4" / "phase4_per_seed_pair_metrics.csv"
OUTPUT_DIR = Path("phase5_bootstrap_stability_results")
RESULT_ZIP_PATH = Path("MentalDistress_Phase5_Bootstrap_Stability_Results.zip")


# 3. Environment and deterministic settings


In [2]:
import json
import os
import platform
import sys
import tempfile
import time

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(0)

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "nbformat": nbformat.__version__,
    "execution_device": "CPU",
    "network_required": False,
}
print(json.dumps(environment, indent=2, sort_keys=True))


{
  "execution_device": "CPU",
  "nbformat": "5.10.4",
  "network_required": false,
  "numpy": "2.5.1",
  "pandas": "3.0.3",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "python": "3.14.4"
}


# 4. Embedded analysis implementation

The complete Phase 3/4 validation helpers and Phase 5 bootstrap implementation are embedded below. The notebook has no dependency on project-local Python modules.


In [3]:
"""Deterministic Phase 3 freeze and Phase 4 boundary-analysis helpers."""

from __future__ import annotations

import csv
import hashlib
import io
import json
from itertools import combinations
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd


LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
EXPECTED_SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
EXPECTED_MODEL_REVISION = "610e9d35d8f2550d713579077242d5e523209cea"
REQUIRED_ARCHIVE_MEMBERS = {
    "artifacts/config.json",
    "artifacts/dataset_integrity.json",
    "artifacts/validation_predictions.csv",
    "artifacts/validation_metrics.json",
    "artifacts/sha256_manifest.csv",
}
METRIC_KEYS = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "mcc",
]
TRAINING_CONFIG_KEYS = [
    "run_mode",
    "max_length",
    "train_batch_size",
    "eval_batch_size",
    "grad_accum_steps",
    "epochs",
    "patience",
    "learning_rate",
    "weight_decay",
    "warmup_ratio",
    "max_grad_norm",
]


def _validate_confusion(confusion, labels):
    matrix = np.asarray(confusion)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("confusion matrix must be square")
    if matrix.shape[0] != len(labels):
        raise ValueError("label count must match confusion matrix dimensions")
    if not np.issubdtype(matrix.dtype, np.number) or not np.isfinite(matrix).all():
        raise ValueError("confusion matrix must contain finite numbers")
    if (matrix < 0).any() or not np.equal(matrix, np.floor(matrix)).all():
        raise ValueError("confusion matrix must contain non-negative integers")
    matrix = matrix.astype(np.int64, copy=False)
    supports = matrix.sum(axis=1)
    if (supports <= 0).any():
        raise ValueError("each class must have positive support")
    if len(labels) != len(set(labels)):
        raise ValueError("labels must be unique")
    return matrix, supports


def compute_pair_metrics(confusion, labels, seed):
    """Compute BCR and error allocation for every unordered label pair."""
    matrix, supports = _validate_confusion(confusion, labels)
    total_errors = int(matrix.sum() - np.trace(matrix))
    rows = []
    for i, j in combinations(range(len(labels)), 2):
        confusion_i_to_j = int(matrix[i, j])
        confusion_j_to_i = int(matrix[j, i])
        pair_errors = confusion_i_to_j + confusion_j_to_i
        bcr = 0.5 * (
            confusion_i_to_j / int(supports[i])
            + confusion_j_to_i / int(supports[j])
        )
        rows.append(
            {
                "seed": int(seed),
                "pair": f"{labels[i]}|{labels[j]}",
                "class_i": labels[i],
                "class_j": labels[j],
                "support_i": int(supports[i]),
                "support_j": int(supports[j]),
                "confusion_i_to_j": confusion_i_to_j,
                "confusion_j_to_i": confusion_j_to_i,
                "pair_errors": pair_errors,
                "bcr": float(bcr),
                "total_errors": total_errors,
                "pair_error_share": float(pair_errors / total_errors) if total_errors else 0.0,
            }
        )
    return pd.DataFrame(rows)


def rank_pairs(pair_metrics):
    """Assign descending average BCR ranks independently within each seed."""
    required = {"seed", "pair", "bcr"}
    missing = required.difference(pair_metrics.columns)
    if missing:
        raise ValueError(f"pair metrics missing columns: {sorted(missing)}")
    frame = pair_metrics.copy()
    if not np.isfinite(frame["bcr"].to_numpy(dtype=float)).all():
        raise ValueError("BCR values must be finite")
    frame["rank"] = frame.groupby("seed", sort=False)["bcr"].rank(
        method="average", ascending=False
    )
    return frame.sort_values(["seed", "rank", "pair"], kind="mergesort").reset_index(drop=True)


def summarize_pairs(per_seed_pairs):
    """Summarize BCRs, ranks, and error allocation across seeds."""
    required = {"pair", "bcr", "rank", "pair_errors", "pair_error_share"}
    missing = required.difference(per_seed_pairs.columns)
    if missing:
        raise ValueError(f"ranked pair metrics missing columns: {sorted(missing)}")

    rows = []
    for pair, group in per_seed_pairs.groupby("pair", sort=True):
        bcr = group["bcr"].to_numpy(dtype=float)
        ranks = group["rank"].to_numpy(dtype=float)
        rows.append(
            {
                "pair": pair,
                "seed_count": int(len(group)),
                "mean_bcr": float(np.mean(bcr)),
                "sample_sd_bcr": float(np.std(bcr, ddof=1)) if len(bcr) > 1 else 0.0,
                "median_bcr": float(np.median(bcr)),
                "bcr_q1": float(np.quantile(bcr, 0.25)),
                "bcr_q3": float(np.quantile(bcr, 0.75)),
                "mean_rank": float(np.mean(ranks)),
                "median_rank": float(np.median(ranks)),
                "top1_count": int(np.sum(ranks <= 1.0)),
                "top1_frequency": float(np.mean(ranks <= 1.0)),
                "top3_count": int(np.sum(ranks <= 3.0)),
                "top3_frequency": float(np.mean(ranks <= 3.0)),
                "mean_pair_errors": float(group["pair_errors"].mean()),
                "mean_pair_error_share": float(group["pair_error_share"].mean()),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["median_rank", "mean_bcr", "pair"], ascending=[True, False, True], kind="mergesort"
    ).reset_index(drop=True)


def _validate_rank_matrix(rank_matrix):
    frame = pd.DataFrame(rank_matrix).copy()
    values = frame.to_numpy(dtype=float)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 2:
        raise ValueError("rank matrix must contain at least two pairs and two seeds")
    if not np.isfinite(values).all():
        raise ValueError("rank matrix must contain only finite values")
    return frame, values


def kendalls_w(rank_matrix):
    """Calculate Kendall's W for pairs (rows) ranked by seeds (columns)."""
    frame, values = _validate_rank_matrix(rank_matrix)
    pair_count, seed_count = values.shape
    expected_rank_sum = pair_count * (pair_count + 1) / 2.0
    column_sums = values.sum(axis=0)
    if not np.allclose(column_sums, expected_rank_sum, rtol=0.0, atol=1e-9):
        raise ValueError("each seed column must contain complete ranks")

    rank_sums = values.sum(axis=1)
    centered_sum_squares = float(
        np.sum((rank_sums - seed_count * (pair_count + 1) / 2.0) ** 2)
    )
    tie_correction = 0.0
    for column in values.T:
        _, counts = np.unique(column, return_counts=True)
        tie_correction += float(np.sum(counts**3 - counts))

    denominator = (
        seed_count**2 * (pair_count**3 - pair_count)
        - seed_count * tie_correction
    )
    if denominator <= 0:
        raise ValueError("Kendall W denominator is not positive")
    coefficient = 12.0 * centered_sum_squares / denominator
    coefficient = float(np.clip(coefficient, 0.0, 1.0))
    return {
        "method": "Kendall coefficient of concordance with tie correction",
        "kendalls_w": coefficient,
        "pair_count": int(pair_count),
        "seed_count": int(seed_count),
        "tie_correction": float(tie_correction),
    }


def pairwise_seed_spearman(rank_matrix):
    """Calculate every pairwise Spearman correlation between seed rankings."""
    frame, _ = _validate_rank_matrix(rank_matrix)
    rows = []
    for seed_a, seed_b in combinations(frame.columns.tolist(), 2):
        a = frame[seed_a].to_numpy(dtype=float)
        b = frame[seed_b].to_numpy(dtype=float)
        rho = float(np.corrcoef(a, b)[0, 1])
        if not np.isfinite(rho):
            raise ValueError("Spearman correlation is not finite")
        rows.append({"seed_a": seed_a, "seed_b": seed_b, "spearman_rho": rho})
    comparisons = pd.DataFrame(rows)
    values = comparisons["spearman_rho"].to_numpy(dtype=float)
    summary = {
        "method": "Pearson correlation applied to complete average-rank vectors",
        "comparison_count": int(len(comparisons)),
        "median_spearman_rho": float(np.median(values)),
        "spearman_rho_q1": float(np.quantile(values, 0.25)),
        "spearman_rho_q3": float(np.quantile(values, 0.75)),
        "minimum_spearman_rho": float(np.min(values)),
        "maximum_spearman_rho": float(np.max(values)),
    }
    return comparisons, summary


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def discover_archives(input_dir):
    input_path = Path(input_dir)
    if not input_path.is_dir():
        raise ValueError(f"input directory does not exist: {input_path}")
    archives = sorted(input_path.glob("mentalroberta_full_seed*_len512_*_ARTIFACTS.zip"))
    if len(archives) != 10:
        raise ValueError(f"expected exactly 10 artifact archives, found {len(archives)}")
    return archives


def validate_archive(path):
    archive_path = Path(path)
    if not archive_path.is_file():
        raise ValueError(f"archive does not exist: {archive_path}")
    try:
        with zipfile.ZipFile(archive_path) as archive:
            corrupt_member = archive.testzip()
            if corrupt_member is not None:
                raise ValueError(f"ZIP integrity failure in {corrupt_member}")
            member_names = set(archive.namelist())
            missing = REQUIRED_ARCHIVE_MEMBERS.difference(member_names)
            if missing:
                raise ValueError(f"archive missing required members: {sorted(missing)}")
            manifest_text = archive.read("artifacts/sha256_manifest.csv").decode("utf-8-sig")
            manifest_rows = list(csv.DictReader(io.StringIO(manifest_text)))
            if not manifest_rows:
                raise ValueError("artifact manifest is empty")
            validated = 0
            for row in manifest_rows:
                relative = row.get("path", "")
                member = f"artifacts/{relative}"
                if member not in member_names:
                    raise ValueError(f"manifest member is absent: {member}")
                payload = archive.read(member)
                expected_bytes = int(row["bytes"])
                if len(payload) != expected_bytes:
                    raise ValueError(f"manifest byte-count mismatch: {member}")
                actual_hash = hashlib.sha256(payload).hexdigest()
                if actual_hash != row["sha256"]:
                    raise ValueError(f"manifest SHA-256 mismatch: {member}")
                validated += 1
    except zipfile.BadZipFile as exc:
        raise ValueError(f"invalid ZIP archive: {archive_path}") from exc
    return {
        "archive_path": archive_path,
        "archive_name": archive_path.name,
        "archive_bytes": int(archive_path.stat().st_size),
        "archive_sha256": sha256_file(archive_path),
        "validated_manifest_members": int(validated),
    }


def _read_json_member(archive, name):
    try:
        return json.loads(archive.read(name).decode("utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError, KeyError) as exc:
        raise ValueError(f"invalid JSON member: {name}") from exc


def validate_predictions(predictions, labels=LABELS, expected_rows=1490):
    required = {"original_row", "true_label", "true_id", "predicted_id", "predicted_label"}
    probability_columns = [f"prob_{label}" for label in labels]
    required.update(probability_columns)
    missing = required.difference(predictions.columns)
    if missing:
        raise ValueError(f"prediction table missing columns: {sorted(missing)}")
    if len(predictions) != expected_rows:
        raise ValueError(f"prediction row count mismatch: expected {expected_rows}, found {len(predictions)}")
    if predictions["original_row"].isna().any() or not predictions["original_row"].is_unique:
        raise ValueError("original_row values must be present and unique")

    expected_label2id = {label: index for index, label in enumerate(labels)}
    true_ids = predictions["true_id"].to_numpy()
    predicted_ids = predictions["predicted_id"].to_numpy()
    if not np.equal(true_ids, np.floor(true_ids)).all() or not np.equal(predicted_ids, np.floor(predicted_ids)).all():
        raise ValueError("label identifiers must be integers")
    true_ids = true_ids.astype(int)
    predicted_ids = predicted_ids.astype(int)
    if not np.isin(true_ids, np.arange(len(labels))).all() or not np.isin(predicted_ids, np.arange(len(labels))).all():
        raise ValueError("label identifiers are outside the locked label order")
    expected_true_labels = np.array([labels[index] for index in true_ids], dtype=object)
    expected_predicted_labels = np.array([labels[index] for index in predicted_ids], dtype=object)
    if not np.array_equal(predictions["true_label"].astype(str).to_numpy(), expected_true_labels):
        raise ValueError("true labels do not match true identifiers")
    if not np.array_equal(predictions["predicted_label"].astype(str).to_numpy(), expected_predicted_labels):
        raise ValueError("predicted labels do not match predicted identifiers")
    if set(predictions["true_label"].unique()) != set(labels):
        raise ValueError("prediction table does not contain the complete locked label set")

    probabilities = predictions[probability_columns].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all():
        raise ValueError("probabilities must be finite")
    if (probabilities < 0).any() or (probabilities > 1).any():
        raise ValueError("probabilities must lie in [0, 1]")
    if not np.allclose(probabilities.sum(axis=1), 1.0, rtol=0.0, atol=1e-5):
        raise ValueError("probabilities must sum to one")
    if not np.array_equal(probabilities.argmax(axis=1), predicted_ids):
        raise ValueError("probability argmax does not match predicted_id")
    return {
        "row_count": int(len(predictions)),
        "original_rows_unique": True,
        "probabilities_valid": True,
        "label_order": list(labels),
        "label2id": expected_label2id,
    }


def _confusion_from_ids(true_ids, predicted_ids, class_count):
    matrix = np.zeros((class_count, class_count), dtype=np.int64)
    np.add.at(matrix, (true_ids, predicted_ids), 1)
    return matrix


def recompute_metrics(predictions, labels=LABELS):
    true_ids = predictions["true_id"].to_numpy(dtype=int)
    predicted_ids = predictions["predicted_id"].to_numpy(dtype=int)
    confusion = _confusion_from_ids(true_ids, predicted_ids, len(labels))
    support = confusion.sum(axis=1).astype(float)
    predicted_support = confusion.sum(axis=0).astype(float)
    true_positive = np.diag(confusion).astype(float)
    precision = np.divide(
        true_positive,
        predicted_support,
        out=np.zeros_like(true_positive),
        where=predicted_support != 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros_like(true_positive),
        where=support != 0,
    )
    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) != 0,
    )
    sample_count = float(confusion.sum())
    correct = float(np.trace(confusion))
    numerator = correct * sample_count - float(np.dot(predicted_support, support))
    denominator = np.sqrt(
        (sample_count**2 - float(np.dot(predicted_support, predicted_support)))
        * (sample_count**2 - float(np.dot(support, support)))
    )
    mcc = numerator / denominator if denominator else 0.0
    per_class = pd.DataFrame(
        {
            "label": labels,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support.astype(int),
        }
    )
    return {
        "accuracy": correct / sample_count,
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "weighted_f1": float(np.average(f1, weights=support)),
        "mcc": float(mcc),
        "confusion_matrix": confusion,
        "per_class": per_class,
    }


def load_run(path, expected_rows=1490, allow_test_access_flag=False):
    archive_validation = validate_archive(path)
    with zipfile.ZipFile(path) as archive:
        config = _read_json_member(archive, "artifacts/config.json")
        integrity = _read_json_member(archive, "artifacts/dataset_integrity.json")
        stored_metrics = _read_json_member(archive, "artifacts/validation_metrics.json")
        predictions = pd.read_csv(io.BytesIO(archive.read("artifacts/validation_predictions.csv")))
    validate_predictions(predictions, LABELS, expected_rows=expected_rows)
    if bool(config.get("test_accessed")) and not allow_test_access_flag:
        raise ValueError("run config records test_accessed=true")
    recomputed = recompute_metrics(predictions, LABELS)
    for key in METRIC_KEYS:
        if key not in stored_metrics:
            raise ValueError(f"stored validation metrics missing {key}")
        if not np.isclose(recomputed[key], stored_metrics[key], rtol=0.0, atol=1e-12):
            raise ValueError(f"recomputed metric mismatch for {key}")
    return {
        **archive_validation,
        "seed": int(config["seed"]),
        "run_id": str(config["run_id"]),
        "config": config,
        "integrity": integrity,
        "stored_metrics": stored_metrics,
        "recomputed_metrics": recomputed,
        "predictions": predictions,
        "warnings": ["full-run repeated-fit reproducibility was not performed"],
    }


def _config_model(config):
    model = config.get("model") or config.get("resolved_model") or {}
    return {
        "model_key": model.get("model_key"),
        "source": model.get("source"),
        "revision": model.get("revision"),
    }


def validate_run_collection(
    runs,
    expected_seeds=EXPECTED_SEEDS,
    expected_rows=1490,
    expected_train_rows=None,
):
    if len(runs) != len(expected_seeds):
        raise ValueError(f"run count mismatch: expected {len(expected_seeds)}, found {len(runs)}")
    observed_seeds = [int(run["seed"]) for run in runs]
    if len(set(observed_seeds)) != len(observed_seeds) or sorted(observed_seeds) != sorted(expected_seeds):
        raise ValueError(f"observed seed set does not match expected seed set: {observed_seeds}")

    reference = sorted(runs, key=lambda item: item["seed"])[0]
    reference_config = reference["config"]
    reference_model = _config_model(reference_config)
    reference_hashes = reference_config.get("data_sha256", {})
    reference_rows = reference["predictions"][["original_row", "true_id", "true_label"]].reset_index(drop=True)
    reference_training = {key: reference_config.get(key) for key in TRAINING_CONFIG_KEYS}
    expected_label2id = {label: index for index, label in enumerate(LABELS)}
    expected_id2label = {str(index): label for index, label in enumerate(LABELS)}

    for run in runs:
        config = run["config"]
        if int(config.get("seed")) != int(run["seed"]):
            raise ValueError("run seed disagrees with config seed")
        model = _config_model(config)
        if model["model_key"] != "mentalroberta" or model["revision"] != EXPECTED_MODEL_REVISION:
            raise ValueError(f"model identity mismatch for seed {run['seed']}: {model}")
        if model != reference_model:
            raise ValueError(f"model source or identity mismatch for seed {run['seed']}")
        if config.get("max_length") != 512:
            raise ValueError(f"max_length mismatch for seed {run['seed']}")
        if bool(config.get("test_accessed")):
            raise ValueError(f"test_accessed must be false for seed {run['seed']}")
        if config.get("label2id") != expected_label2id:
            raise ValueError(f"label mapping mismatch for seed {run['seed']}")
        normalized_id2label = {str(key): value for key, value in config.get("id2label", {}).items()}
        if normalized_id2label != expected_id2label:
            raise ValueError(f"inverse label mapping mismatch for seed {run['seed']}")
        if config.get("data_sha256", {}) != reference_hashes:
            raise ValueError("dataset hashes do not match across seeds")
        integrity = run.get("integrity", {})
        if integrity.get("sha256") != config.get("data_sha256"):
            raise ValueError(f"dataset integrity hashes disagree with config for seed {run['seed']}")
        if bool(integrity.get("test_accessed", False)):
            raise ValueError(f"dataset integrity records test_accessed=true for seed {run['seed']}")
        integrity_rows = integrity.get("rows", {})
        if int(config.get("validation_rows_used", -1)) != expected_rows:
            raise ValueError(f"validation row count in config mismatch for seed {run['seed']}")
        if int(integrity_rows.get("validation", -1)) != expected_rows:
            raise ValueError(f"validation row count in integrity file mismatch for seed {run['seed']}")
        if int(integrity_rows.get("train", -1)) != int(config.get("train_rows_used", -2)):
            raise ValueError(f"training row counts disagree for seed {run['seed']}")
        if expected_train_rows is not None and int(config.get("train_rows_used", -1)) != expected_train_rows:
            raise ValueError(f"training row count mismatch for seed {run['seed']}")
        training = {key: config.get(key) for key in TRAINING_CONFIG_KEYS}
        if training != reference_training:
            raise ValueError(f"training configuration mismatch for seed {run['seed']}")
        predictions = run["predictions"]
        validate_predictions(predictions, LABELS, expected_rows=expected_rows)
        rows = predictions[["original_row", "true_id", "true_label"]].reset_index(drop=True)
        if not rows.equals(reference_rows):
            raise ValueError(f"validation row identities or true labels mismatch for seed {run['seed']}")
    return {
        "run_count": int(len(runs)),
        "observed_seeds": sorted(observed_seeds),
        "model": reference_model,
        "data_sha256": dict(reference_hashes),
        "training_config": reference_training,
        "validation_rows": int(expected_rows),
        "train_rows": int(reference_config.get("train_rows_used", 0)),
        "test_accessed": False,
        "all_hard_validations_pass": True,
    }


def _json_scalar(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"not JSON serializable: {type(value).__name__}")


def build_phase3_freeze(runs, source_paths=None, optional_summary=None):
    """Reconstruct the frozen Phase 3 inventory and aggregate metrics."""
    if not runs:
        raise ValueError("at least one run is required")
    ordered = sorted(runs, key=lambda item: item["seed"])
    expected = [run["seed"] for run in ordered]
    collection = validate_run_collection(
        ordered,
        expected_seeds=expected,
        expected_rows=len(ordered[0]["predictions"]),
    )
    inventory_rows = []
    metric_rows = []
    for run in ordered:
        inventory_rows.append(
            {
                "seed": int(run["seed"]),
                "run_id": run["run_id"],
                "archive_name": run["archive_name"],
                "archive_bytes": int(run["archive_bytes"]),
                "archive_sha256": run["archive_sha256"],
                "validated_manifest_members": int(run["validated_manifest_members"]),
                "validation_rows": int(len(run["predictions"])),
                "test_accessed": bool(run["config"].get("test_accessed")),
            }
        )
        row = {"seed": int(run["seed"]), "run_id": run["run_id"]}
        for key, value in run["stored_metrics"].items():
            if isinstance(value, (int, float)):
                row[key] = float(value)
        metric_rows.append(row)
    run_inventory = pd.DataFrame(inventory_rows).sort_values("seed").reset_index(drop=True)
    per_seed_metrics = pd.DataFrame(metric_rows).sort_values("seed").reset_index(drop=True)
    aggregate_rows = []
    for metric in [*METRIC_KEYS, "loss"]:
        values = per_seed_metrics[metric].to_numpy(dtype=float)
        aggregate_rows.append(
            {
                "metric": metric,
                "seed_count": int(len(values)),
                "mean": float(np.mean(values)),
                "sample_sd": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
                "minimum": float(np.min(values)),
                "maximum": float(np.max(values)),
            }
        )
    aggregate_metrics = pd.DataFrame(aggregate_rows)

    source_hashes = {}
    for name, path in sorted((source_paths or {}).items()):
        candidate = Path(path)
        source_hashes[name] = {
            "path": str(candidate),
            "sha256": sha256_file(candidate) if candidate.is_file() else None,
            "present": candidate.is_file(),
        }
    optional_summary_check = {"supplied": optional_summary is not None, "matched": None}
    if optional_summary is not None:
        macro = aggregate_metrics.set_index("metric").loc["macro_f1"]
        checks = []
        mappings = {
            "validation_macro_f1_mean": float(macro["mean"]),
            "validation_macro_f1_sample_sd": float(macro["sample_sd"]),
            "validation_accuracy_mean": float(
                aggregate_metrics.set_index("metric").loc["accuracy", "mean"]
            ),
            "completed_seeds": len(ordered),
            "seeds": [int(run["seed"]) for run in ordered],
            "model_key": collection["model"]["model_key"],
            "model_source": collection["model"]["source"],
            "model_revision": collection["model"]["revision"],
            "data_sha256": collection["data_sha256"],
            "test_accessed": False,
        }
        mappings.update(collection["training_config"])
        for key, expected_value in mappings.items():
            if key in optional_summary:
                observed = optional_summary[key]
                if isinstance(expected_value, float):
                    checks.append(bool(np.isclose(observed, expected_value, rtol=0.0, atol=1e-12)))
                else:
                    checks.append(observed == expected_value)
        optional_summary_check["matched"] = bool(checks and all(checks))
        optional_summary_check["checked_fields"] = sorted(
            key for key in mappings if key in optional_summary
        )
        if not optional_summary_check["matched"]:
            raise ValueError("optional Phase 3 summary does not match reconstructed results")

    reference = ordered[0]["config"]
    manifest = {
        "phase": "Phase 3 — Ten-Seed Principal Global Model",
        "status": "PASS",
        "created_utc": "2026-08-22T00:00:00Z",
        "creation_time_policy": "Fixed protocol-date timestamp for deterministic packaging",
        "expected_seeds": list(collection["observed_seeds"]),
        "observed_seeds": list(collection["observed_seeds"]),
        "completed_seeds": int(len(ordered)),
        "model": collection["model"],
        "label_order": list(LABELS),
        "training_config": collection["training_config"],
        "data_sha256": collection["data_sha256"],
        "train_rows": int(reference.get("train_rows_used", 0)),
        "validation_rows": int(reference.get("validation_rows_used", len(ordered[0]["predictions"]))),
        "input_archives": inventory_rows,
        "aggregate_metrics": {
            row["metric"]: {
                "mean": row["mean"],
                "sample_sd": row["sample_sd"],
                "minimum": row["minimum"],
                "maximum": row["maximum"],
            }
            for row in aggregate_rows
        },
        "optional_summary_check": optional_summary_check,
        "source_files": source_hashes,
        "model_archives_local": False,
        "model_archive_policy": "Model archives were intentionally not included in this local freeze.",
        "test_accessed": False,
        "all_hard_validations_pass": True,
    }
    return {
        "run_inventory": run_inventory,
        "per_seed_metrics": per_seed_metrics,
        "aggregate_metrics": aggregate_metrics,
        "manifest": manifest,
        "warnings": sorted(set(warning for run in ordered for warning in run.get("warnings", []))),
    }


def build_phase4_analysis(runs):
    """Build all Phase 4 per-seed and cross-seed boundary statistics."""
    ordered = sorted(runs, key=lambda item: item["seed"])
    pair_frames = []
    confusion_rows = []
    for run in ordered:
        seed = int(run["seed"])
        confusion = recompute_metrics(run["predictions"], LABELS)["confusion_matrix"]
        for true_index, true_label in enumerate(LABELS):
            for predicted_index, predicted_label in enumerate(LABELS):
                confusion_rows.append(
                    {
                        "seed": seed,
                        "true_label": true_label,
                        "predicted_label": predicted_label,
                        "count": int(confusion[true_index, predicted_index]),
                    }
                )
        pair_frames.append(rank_pairs(compute_pair_metrics(confusion, LABELS, seed)))
    per_seed_pairs = pd.concat(pair_frames, ignore_index=True).sort_values(
        ["seed", "rank", "pair"], kind="mergesort"
    ).reset_index(drop=True)
    stability = summarize_pairs(per_seed_pairs)
    rank_matrix = per_seed_pairs.pivot(index="pair", columns="seed", values="rank")
    rank_matrix = rank_matrix.reindex(sorted(rank_matrix.index)).reindex(
        columns=[run["seed"] for run in ordered]
    )
    rank_matrix.columns = [str(column) for column in rank_matrix.columns]
    agreement = kendalls_w(rank_matrix)
    spearman_rows, spearman_summary = pairwise_seed_spearman(rank_matrix)
    agreement["pairwise_spearman"] = spearman_summary

    candidates = stability[stability["top3_count"] >= 7].copy()
    candidates = candidates.sort_values(
        ["median_rank", "mean_bcr", "pair"], ascending=[True, False, True], kind="mergesort"
    )
    criterion1 = {
        "criterion": "top-three rank in at least 7 of 10 principal seeds",
        "threshold": 7,
        "candidate_pairs": candidates["pair"].tolist(),
        "candidate_count": int(len(candidates)),
        "final_hard_pairs_frozen": False,
        "phase5_required": True,
        "phase5_bootstrap_top3_frequency_threshold": 0.70,
        "interpretation": (
            "These pairs satisfy only criterion 1 and remain provisional until the "
            "predeclared Phase 5 bootstrap criterion is evaluated."
        ),
    }
    return {
        "per_seed_confusion_matrices": pd.DataFrame(confusion_rows),
        "per_seed_pair_metrics": per_seed_pairs,
        "pair_stability_summary": stability,
        "seed_rank_matrix": rank_matrix,
        "pairwise_seed_spearman": spearman_rows,
        "rank_agreement": agreement,
        "criterion1_candidates": criterion1,
    }


def _write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=_json_scalar) + "\n",
        encoding="utf-8",
    )


def _write_dataframe(path, frame):
    frame.to_csv(path, index=False, float_format="%.17g", lineterminator="\n")


def _write_figures(phase4, figures_dir):
    import matplotlib.pyplot as plt

    figures_dir.mkdir(parents=True, exist_ok=True)
    summary = phase4["pair_stability_summary"].sort_values("mean_bcr")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(summary["pair"], summary["mean_bcr"], xerr=summary["sample_sd_bcr"], capsize=3)
    ax.set_xlabel("Mean bidirectional confusion rate")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 mean BCR across ten principal seeds")
    fig.tight_layout()
    fig.savefig(figures_dir / "mean_bcr_by_pair.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    error_summary = phase4["pair_stability_summary"].sort_values("mean_pair_error_share")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(error_summary["pair"], error_summary["mean_pair_error_share"])
    ax.set_xlabel("Mean share of all validation errors")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 error concentration by pair")
    fig.tight_layout()
    fig.savefig(figures_dir / "error_share_by_pair.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    rank_matrix = phase4["seed_rank_matrix"]
    fig, ax = plt.subplots(figsize=(10, 7))
    image = ax.imshow(rank_matrix.to_numpy(dtype=float), aspect="auto", cmap="viridis_r", vmin=1, vmax=10)
    ax.set_xticks(range(len(rank_matrix.columns)), rank_matrix.columns)
    ax.set_yticks(range(len(rank_matrix.index)), rank_matrix.index)
    ax.set_xlabel("Seed")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 BCR ranks (lower is harder)")
    fig.colorbar(image, ax=ax, label="Rank")
    fig.tight_layout()
    fig.savefig(figures_dir / "pair_rank_heatmap.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def write_sha256_manifest(root, output_path=None):
    root = Path(root)
    output_path = Path(output_path) if output_path else root / "sha256_manifest.csv"
    rows = []
    for path in sorted(candidate for candidate in root.rglob("*") if candidate.is_file() and candidate != output_path):
        rows.append(
            {
                "path": path.relative_to(root).as_posix(),
                "sha256": sha256_file(path),
                "bytes": int(path.stat().st_size),
            }
        )
    pd.DataFrame(rows, columns=["path", "sha256", "bytes"]).to_csv(
        output_path, index=False, lineterminator="\n"
    )
    return rows


def write_results(phase3, phase4, output_dir, create_figures=True):
    """Write the approved Phase 3 freeze and Phase 4 artifact tree."""
    output_dir = Path(output_dir)
    phase3_dir = output_dir / "phase3_freeze"
    phase4_dir = output_dir / "phase4"
    figures_dir = phase4_dir / "figures"
    phase3_dir.mkdir(parents=True, exist_ok=True)
    phase4_dir.mkdir(parents=True, exist_ok=True)

    _write_dataframe(phase3_dir / "phase3_run_inventory.csv", phase3["run_inventory"])
    _write_dataframe(phase3_dir / "phase3_per_seed_metrics.csv", phase3["per_seed_metrics"])
    _write_dataframe(phase3_dir / "phase3_aggregate_metrics.csv", phase3["aggregate_metrics"])
    _write_json(phase3_dir / "phase3_freeze_manifest.json", phase3["manifest"])
    macro = phase3["aggregate_metrics"].set_index("metric").loc["macro_f1"]
    accuracy = phase3["aggregate_metrics"].set_index("metric").loc["accuracy"]
    phase3_report = (
        "# Phase 3 Freeze Report\n\n"
        f"Status: **{phase3['manifest']['status']}**\n\n"
        f"Completed seeds: {phase3['manifest']['completed_seeds']}\n\n"
        f"Validation Macro-F1: {macro['mean']:.9f} ± {macro['sample_sd']:.9f} (sample SD)\n\n"
        f"Validation accuracy: {accuracy['mean']:.9f} ± {accuracy['sample_sd']:.9f} (sample SD)\n\n"
        "The locked test was not accessed. Model archives were intentionally not included locally.\n"
    )
    (phase3_dir / "phase3_freeze_report.md").write_text(phase3_report, encoding="utf-8")

    _write_dataframe(
        phase4_dir / "phase4_per_seed_confusion_matrices.csv",
        phase4["per_seed_confusion_matrices"],
    )
    _write_dataframe(
        phase4_dir / "phase4_per_seed_pair_metrics.csv",
        phase4["per_seed_pair_metrics"],
    )
    _write_dataframe(
        phase4_dir / "phase4_pair_stability_summary.csv",
        phase4["pair_stability_summary"],
    )
    rank_export = phase4["seed_rank_matrix"].reset_index()
    _write_dataframe(phase4_dir / "phase4_seed_rank_matrix.csv", rank_export)
    _write_dataframe(
        phase4_dir / "phase4_pairwise_seed_spearman.csv",
        phase4["pairwise_seed_spearman"],
    )
    _write_json(phase4_dir / "phase4_rank_agreement.json", phase4["rank_agreement"])
    _write_json(
        phase4_dir / "phase4_criterion1_candidates.json",
        phase4["criterion1_candidates"],
    )
    top_rows = phase4["pair_stability_summary"].head(3)
    top_lines = "\n".join(
        f"- {row.pair}: mean BCR {row.mean_bcr:.6f}, top-three count {int(row.top3_count)}/10"
        for row in top_rows.itertuples()
    )
    phase4_report = (
        "# Phase 4 Residual Boundary Report\n\n"
        "This report identifies **provisional criterion-1 candidates only**. "
        "No final hard pair is frozen here; Phase 5 bootstrap validation is mandatory.\n\n"
        "## Leading observed pairs\n\n"
        f"{top_lines}\n\n"
        f"Kendall's W: {phase4['rank_agreement']['kendalls_w']:.6f}\n"
    )
    (phase4_dir / "phase4_report.md").write_text(phase4_report, encoding="utf-8")
    if create_figures:
        _write_figures(phase4, figures_dir)

    execution_summary = {
        "status": "PASS",
        "archive_count": int(len(phase3["run_inventory"])),
        "completed_seeds": int(phase3["manifest"]["completed_seeds"]),
        "all_hard_validations_pass": True,
        "test_accessed": False,
        "validation_macro_f1_mean": float(macro["mean"]),
        "validation_macro_f1_sample_sd": float(macro["sample_sd"]),
        "validation_accuracy_mean": float(accuracy["mean"]),
        "kendalls_w": float(phase4["rank_agreement"]["kendalls_w"]),
        "criterion1_candidates": phase4["criterion1_candidates"]["candidate_pairs"],
        "final_hard_pairs_frozen": False,
        "phase5_required": True,
        "warnings": list(phase3.get("warnings", [])),
    }
    _write_json(output_dir / "phase3_phase4_execution_summary.json", execution_summary)
    manifest_rows = write_sha256_manifest(output_dir)
    return {
        "output_dir": output_dir,
        "manifest_entries": int(len(manifest_rows)),
        "execution_summary": execution_summary,
    }


def write_deterministic_zip(source_dir, zip_path, prefix="phase4_boundary_analysis_results"):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    fixed_time = (2020, 1, 1, 0, 0, 0)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(candidate for candidate in source_dir.rglob("*") if candidate.is_file()):
            member = (Path(prefix) / path.relative_to(source_dir)).as_posix()
            info = zipfile.ZipInfo(member, fixed_time)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, path.read_bytes())
    return zip_path


In [4]:
"""Deterministic Phase 5 validation-bootstrap stability analysis."""

from __future__ import annotations

from itertools import combinations
import html
import json
from pathlib import Path

import numpy as np
import pandas as pd



BOOTSTRAP_REPLICATES = 2000
MASTER_BOOTSTRAP_SEED = 20260822
CRITERION1_COUNT = 7
CRITERION2_FREQUENCY = 0.70
K_MAX = 3


def make_seed_rng(master_seed, model_seed):
    """Return the locked deterministic random generator for one model seed."""
    sequence = np.random.SeedSequence([int(master_seed), int(model_seed)])
    return np.random.default_rng(sequence)


def stratified_bootstrap_indices(true_ids, class_ids, rng):
    """Sample with replacement within each true class, preserving support."""
    values = np.asarray(true_ids, dtype=int)
    sampled = []
    for class_id in class_ids:
        candidates = np.flatnonzero(values == int(class_id))
        if len(candidates) == 0:
            raise ValueError(f"true class {class_id} has zero support")
        sampled.append(rng.choice(candidates, size=len(candidates), replace=True))
    return np.concatenate(sampled).astype(np.int64, copy=False)


def compute_bcr_vector(confusion, labels=LABELS):
    """Compute Phase 4 BCRs in deterministic unordered-pair order."""
    matrix = np.asarray(confusion)
    if matrix.shape != (len(labels), len(labels)):
        raise ValueError("confusion matrix shape does not match labels")
    if not np.issubdtype(matrix.dtype, np.number) or not np.isfinite(matrix).all():
        raise ValueError("confusion matrix must contain finite numbers")
    if (matrix < 0).any() or not np.equal(matrix, np.floor(matrix)).all():
        raise ValueError("confusion matrix must contain non-negative integers")
    supports = matrix.sum(axis=1).astype(float)
    if (supports <= 0).any():
        raise ValueError("each class must have positive support")
    pairs = []
    values = []
    for i, j in combinations(range(len(labels)), 2):
        pairs.append((i, j, f"{labels[i]}|{labels[j]}"))
        values.append(0.5 * (matrix[i, j] / supports[i] + matrix[j, i] / supports[j]))
    return pairs, np.asarray(values, dtype=float)


def rank_bcr_vector(bcr):
    """Return descending average ranks for a one-dimensional BCR vector."""
    values = np.asarray(bcr, dtype=float)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("BCR vector must be non-empty, one-dimensional, and finite")
    return pd.Series(values).rank(method="average", ascending=False).to_numpy(dtype=float)


def bootstrap_seed(run, labels=LABELS, replicate_count=BOOTSTRAP_REPLICATES,
                   master_seed=MASTER_BOOTSTRAP_SEED):
    """Run the locked stratified bootstrap for one frozen model seed."""
    if int(replicate_count) <= 0:
        raise ValueError("replicate_count must be positive")
    predictions = run["predictions"]
    true_ids = predictions["true_id"].to_numpy(dtype=int)
    predicted_ids = predictions["predicted_id"].to_numpy(dtype=int)
    class_ids = list(range(len(labels)))
    pair_definitions = list(combinations(class_ids, 2))
    pair_count = len(pair_definitions)
    bcr_values = np.empty((int(replicate_count), pair_count), dtype=np.float64)
    rank_values = np.empty_like(bcr_values)
    rng = make_seed_rng(master_seed, int(run["seed"]))

    for replicate_index in range(int(replicate_count)):
        sampled = stratified_bootstrap_indices(true_ids, class_ids, rng)
        confusion = _confusion_from_ids(
            true_ids[sampled], predicted_ids[sampled], len(labels)
        )
        _, bcr = compute_bcr_vector(confusion, labels)
        bcr_values[replicate_index] = bcr
        rank_values[replicate_index] = rank_bcr_vector(bcr)

    replicate_numbers = np.repeat(np.arange(1, int(replicate_count) + 1), pair_count)
    pair_numbers = np.tile(np.arange(pair_count), int(replicate_count))
    class_a = [labels[i] for i, _ in pair_definitions]
    class_b = [labels[j] for _, j in pair_definitions]
    pair_names = [f"{a}|{b}" for a, b in zip(class_a, class_b)]
    flat_ranks = rank_values.ravel()
    return pd.DataFrame(
        {
            "model_seed": int(run["seed"]),
            "bootstrap_replicate": replicate_numbers,
            "pair": np.asarray(pair_names, dtype=object)[pair_numbers],
            "class_a": np.asarray(class_a, dtype=object)[pair_numbers],
            "class_b": np.asarray(class_b, dtype=object)[pair_numbers],
            "bcr": bcr_values.ravel(),
            "rank": flat_ranks,
            "is_top1": flat_ranks <= 1.0,
            "is_top3": flat_ranks <= 3.0,
        }
    )


def summarize_seed_pairs(replicates):
    """Summarize bootstrap BCR and rank behavior for each seed and pair."""
    required = {"model_seed", "pair", "bcr", "rank", "is_top1", "is_top3"}
    missing = required.difference(replicates.columns)
    if missing:
        raise ValueError(f"replicate table missing columns: {sorted(missing)}")
    rows = []
    for (seed, pair), group in replicates.groupby(["model_seed", "pair"], sort=True):
        bcr = group["bcr"].to_numpy(dtype=float)
        ranks = group["rank"].to_numpy(dtype=float)
        top1 = group["is_top1"].to_numpy(dtype=bool)
        top3 = group["is_top3"].to_numpy(dtype=bool)
        rows.append(
            {
                "model_seed": int(seed),
                "pair": pair,
                "bootstrap_replicates": int(len(group)),
                "bcr_mean": float(np.mean(bcr)),
                "bcr_sample_sd": float(np.std(bcr, ddof=1)) if len(bcr) > 1 else 0.0,
                "bcr_median": float(np.median(bcr)),
                "bcr_q1": float(np.quantile(bcr, 0.25, method="linear")),
                "bcr_q3": float(np.quantile(bcr, 0.75, method="linear")),
                "bcr_ci_lower": float(np.quantile(bcr, 0.025, method="linear")),
                "bcr_ci_upper": float(np.quantile(bcr, 0.975, method="linear")),
                "top1_count": int(np.sum(top1)),
                "top1_frequency": float(np.mean(top1)),
                "top3_count": int(np.sum(top3)),
                "top3_frequency": float(np.mean(top3)),
                "median_rank": float(np.median(ranks)),
            }
        )
    return pd.DataFrame(rows).sort_values(["model_seed", "pair"], kind="mergesort").reset_index(drop=True)


def summarize_pairs_equal_seed(seed_summary, replicates):
    """Build the equal-seed primary summary and labelled pooled descriptors."""
    rows = []
    for pair, seed_group in seed_summary.groupby("pair", sort=True):
        pooled = replicates.loc[replicates["pair"] == pair, "bcr"].to_numpy(dtype=float)
        rows.append(
            {
                "pair": pair,
                "seed_count": int(len(seed_group)),
                "mean_seed_bcr_mean": float(seed_group["bcr_mean"].mean()),
                "mean_seed_bcr_sample_sd": float(seed_group["bcr_sample_sd"].mean()),
                "mean_seed_top1_frequency": float(seed_group["top1_frequency"].mean()),
                "mean_seed_top3_frequency": float(seed_group["top3_frequency"].mean()),
                "median_seed_median_rank": float(seed_group["median_rank"].median()),
                "pooled_observation_count": int(len(pooled)),
                "pooled_bcr_mean": float(np.mean(pooled)),
                "pooled_bcr_median": float(np.median(pooled)),
                "pooled_bcr_ci_lower": float(np.quantile(pooled, 0.025, method="linear")),
                "pooled_bcr_ci_upper": float(np.quantile(pooled, 0.975, method="linear")),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["mean_seed_top3_frequency", "mean_seed_bcr_mean", "pair"],
        ascending=[False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)


def build_rank_distribution(replicates):
    """Count every observed fractional rank by model seed and pair."""
    counts = (
        replicates.groupby(["model_seed", "pair", "rank"], sort=True)
        .size()
        .rename("count")
        .reset_index()
    )
    totals = counts.groupby(["model_seed", "pair"])["count"].transform("sum")
    counts["frequency"] = counts["count"] / totals
    return counts


def select_hard_pairs(original_summary, pooled_summary, criterion1_count=CRITERION1_COUNT,
                      criterion2_frequency=CRITERION2_FREQUENCY, k_max=K_MAX):
    """Apply both frozen criteria and deterministic ordering to select H."""
    required_original = {"pair", "top3_count", "median_rank", "mean_bcr"}
    required_pooled = {"pair", "mean_seed_top3_frequency"}
    if required_original.difference(original_summary.columns):
        raise ValueError("original summary is missing selection columns")
    if required_pooled.difference(pooled_summary.columns):
        raise ValueError("pooled summary is missing selection columns")
    decisions = original_summary[list(required_original)].merge(
        pooled_summary[list(required_pooled)], on="pair", how="inner", validate="one_to_one"
    )
    if len(decisions) != len(original_summary) or int(k_max) < 0:
        raise ValueError("selection inputs do not contain the same pairs or k_max is invalid")
    decisions["criterion1_pass"] = decisions["top3_count"] >= int(criterion1_count)
    decisions["criterion2_pass"] = (
        decisions["mean_seed_top3_frequency"] >= float(criterion2_frequency)
    )
    decisions["qualifies"] = decisions["criterion1_pass"] & decisions["criterion2_pass"]
    decisions = decisions.sort_values(
        ["qualifies", "median_rank", "mean_bcr", "mean_seed_top3_frequency", "pair"],
        ascending=[False, True, False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    selected = decisions.loc[decisions["qualifies"], "pair"].head(int(k_max)).tolist()
    return {
        "phase": "Phase 5 — Bootstrap Stability Validation",
        "status": "FROZEN",
        "H": selected,
        "K": int(len(selected)),
        "K_max": int(k_max),
        "criterion1": {
            "definition": "original Phase 4 top-three rank count across ten model seeds",
            "minimum_count": int(criterion1_count),
        },
        "criterion2": {
            "definition": "mean of seed-specific bootstrap top-three frequencies",
            "minimum_frequency": float(criterion2_frequency),
        },
        "tie_breaking": [
            "lower median original Phase 4 rank",
            "higher mean original Phase 4 BCR",
            "higher mean seed-specific bootstrap top-three frequency",
            "lexicographic pair name",
        ],
        "pair_decisions": decisions.to_dict(orient="records"),
        "test_accessed": False,
    }


def _original_phase4(runs):
    phase4 = build_phase4_analysis(runs)
    original = phase4["per_seed_pair_metrics"].rename(
        columns={"seed": "model_seed", "class_i": "class_a", "class_j": "class_b"}
    )
    original["is_top1"] = original["rank"] <= 1.0
    original["is_top3"] = original["rank"] <= 3.0
    return phase4, original


def run_phase5(runs, labels=LABELS, replicate_count=BOOTSTRAP_REPLICATES,
               master_seed=MASTER_BOOTSTRAP_SEED):
    """Run Phase 5 for validated frozen runs and return every result table."""
    if not runs:
        raise ValueError("at least one run is required")
    ordered = sorted(runs, key=lambda item: int(item["seed"]))
    if len({int(run["seed"]) for run in ordered}) != len(ordered):
        raise ValueError("model seeds must be unique")
    phase4, original = _original_phase4(ordered)
    replicate_frames = [
        bootstrap_seed(run, labels, replicate_count, master_seed) for run in ordered
    ]
    replicates = pd.concat(replicate_frames, ignore_index=True)
    seed_summary = summarize_seed_pairs(replicates)
    pooled_summary = summarize_pairs_equal_seed(seed_summary, replicates)
    rank_distribution = build_rank_distribution(replicates)
    selection = select_hard_pairs(phase4["pair_stability_summary"], pooled_summary)
    selection.update(
        {
            "bootstrap_replicates_per_seed": int(replicate_count),
            "master_bootstrap_seed": int(master_seed),
            "model_seeds": [int(run["seed"]) for run in ordered],
            "input_artifacts": [
                {
                    "model_seed": int(run["seed"]),
                    "archive_name": run.get("archive_name"),
                    "archive_sha256": run.get("archive_sha256"),
                }
                for run in ordered
            ],
        }
    )
    protocol = {
        "phase": 5,
        "bootstrap_replicates_per_seed": int(replicate_count),
        "master_bootstrap_seed": int(master_seed),
        "per_seed_rng": "numpy.random.default_rng(numpy.random.SeedSequence([master_seed, model_seed]))",
        "resampling": "with replacement within each true class; original class supports preserved",
        "pair_count": int(len(pooled_summary)),
        "rank_method": "descending BCR with average ranks for exact ties",
        "top1_rule": "rank <= 1",
        "top3_rule": "rank <= 3",
        "quantile_method": "linear",
        "seed_weighting": "equal",
        "criterion1_minimum_seed_count": CRITERION1_COUNT,
        "criterion2_minimum_mean_seed_top3_frequency": CRITERION2_FREQUENCY,
        "K_max": K_MAX,
        "execution_device": "CPU",
        "test_accessed": False,
    }
    return {
        "protocol": protocol,
        "original_phase4": phase4,
        "original_pair_metrics": original,
        "per_replicate_pair_metrics": replicates,
        "seed_pair_summary": seed_summary,
        "pair_pooled_summary": pooled_summary,
        "rank_distribution": rank_distribution,
        "hard_pair_selection": selection,
    }


def crosscheck_phase4(original_pair_metrics, frozen_csv, atol=1e-12):
    """Require exact identifiers/flags and near-exact Phase 4 numeric reproduction."""
    frozen = pd.read_csv(frozen_csv).rename(
        columns={"seed": "model_seed", "class_i": "class_a", "class_j": "class_b"}
    )
    observed = original_pair_metrics.copy()
    keys = ["model_seed", "pair"]
    frozen = frozen.sort_values(keys).reset_index(drop=True)
    observed = observed.sort_values(keys).reset_index(drop=True)
    if not observed[keys].equals(frozen[keys]):
        raise ValueError("Phase 4 cross-check identifier mismatch")
    for column in ["bcr", "rank"]:
        if not np.allclose(observed[column], frozen[column], rtol=0.0, atol=float(atol)):
            raise ValueError(f"Phase 4 cross-check mismatch for {column}")
    for column in ["class_a", "class_b"]:
        if not observed[column].equals(frozen[column]):
            raise ValueError(f"Phase 4 cross-check mismatch for {column}")
    return {
        "status": "PASS",
        "row_count": int(len(observed)),
        "absolute_tolerance": float(atol),
        "checked_numeric_fields": ["bcr", "rank"],
        "checked_identifier_fields": keys + ["class_a", "class_b"],
    }


def validate_phase5_results(results, seed_count, replicate_count):
    """Assert the locked post-run invariants and return a compact audit."""
    long = results["per_replicate_pair_metrics"]
    seed_summary = results["seed_pair_summary"]
    pooled = results["pair_pooled_summary"]
    expected_rows = int(seed_count) * int(replicate_count) * 10
    if len(long) != expected_rows:
        raise ValueError(f"expected {expected_rows} replicate-pair rows, found {len(long)}")
    per_seed = long.groupby("model_seed")["bootstrap_replicate"].nunique()
    if len(per_seed) != int(seed_count) or not (per_seed == int(replicate_count)).all():
        raise ValueError("bootstrap replicate count mismatch")
    per_pair = long.groupby("pair").size()
    if len(per_pair) != 10 or not (per_pair == int(seed_count) * int(replicate_count)).all():
        raise ValueError("per-pair observation count mismatch")
    if len(seed_summary) != int(seed_count) * 10 or len(pooled) != 10:
        raise ValueError("summary row count mismatch")
    if not np.isfinite(long[["bcr", "rank"]].to_numpy(dtype=float)).all():
        raise ValueError("non-finite bootstrap metric")
    if not np.array_equal(long["is_top1"].to_numpy(dtype=bool), long["rank"].to_numpy() <= 1):
        raise ValueError("top-1 flags disagree with ranks")
    if not np.array_equal(long["is_top3"].to_numpy(dtype=bool), long["rank"].to_numpy() <= 3):
        raise ValueError("top-3 flags disagree with ranks")
    frequency_columns = [column for column in seed_summary if column.endswith("frequency")]
    frequency_columns += [column for column in pooled if column.endswith("frequency")]
    for column in frequency_columns:
        values = (seed_summary[column] if column in seed_summary else pooled[column]).to_numpy(dtype=float)
        if ((values < 0) | (values > 1)).any():
            raise ValueError(f"frequency outside [0, 1]: {column}")
    return {
        "status": "PASS",
        "seed_count": int(seed_count),
        "bootstrap_replicates_per_seed": int(replicate_count),
        "pair_count": 10,
        "replicate_pair_rows": int(len(long)),
        "seed_pair_summary_rows": int(len(seed_summary)),
        "pair_pooled_summary_rows": int(len(pooled)),
        "finite_metrics": True,
        "rank_flags_consistent": True,
    }


def _write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=_json_scalar) + "\n",
        encoding="utf-8",
    )


def _write_dataframe(path, frame):
    frame.to_csv(path, index=False, float_format="%.17g", lineterminator="\n")


def _write_figures(results, figures_dir):
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)

    def svg_document(title, body, width=1000, height=700):
        return (
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
            f'viewBox="0 0 {width} {height}">\n'
            '<rect width="100%" height="100%" fill="white"/>\n'
            '<style>text{font-family:Arial,sans-serif;fill:#172033}.title{font-size:24px;font-weight:700}'
            '.label{font-size:14px}.small{font-size:12px}.axis{stroke:#65728a;stroke-width:1}</style>\n'
            f'<text x="500" y="34" text-anchor="middle" class="title">{html.escape(title)}</text>\n'
            f'{body}\n</svg>\n'
        )

    summary = results["pair_pooled_summary"].sort_values(
        "mean_seed_top3_frequency", ascending=False, kind="mergesort"
    )
    bar_lines = ['<line x1="300" y1="70" x2="300" y2="630" class="axis"/>']
    threshold_x = 300 + 620 * CRITERION2_FREQUENCY
    bar_lines.append(
        f'<line x1="{threshold_x:.2f}" y1="70" x2="{threshold_x:.2f}" y2="630" '
        'stroke="#b42318" stroke-width="2" stroke-dasharray="7,5"/>'
    )
    bar_lines.append(
        f'<text x="{threshold_x + 5:.2f}" y="65" class="small">0.70 qualification threshold</text>'
    )
    for row_index, row in enumerate(summary.itertuples(index=False)):
        y = 82 + row_index * 52
        width = 620 * float(row.mean_seed_top3_frequency)
        bar_lines.append(f'<text x="290" y="{y + 20}" text-anchor="end" class="label">{html.escape(row.pair)}</text>')
        bar_lines.append(f'<rect x="300" y="{y}" width="{width:.2f}" height="28" fill="#2f6f9f"/>')
        bar_lines.append(f'<text x="{305 + width:.2f}" y="{y + 20}" class="label">{row.mean_seed_top3_frequency:.3f}</text>')
    (figures_dir / "bootstrap_top3_frequency_by_pair.svg").write_text(
        svg_document("Phase 5 bootstrap top-3 stability", "\n".join(bar_lines)), encoding="utf-8"
    )

    ordered = results["pair_pooled_summary"].sort_values(
        "pooled_bcr_mean", ascending=False, kind="mergesort"
    )
    max_bcr = max(float(ordered["pooled_bcr_ci_upper"].max()), 1e-12)
    interval_lines = ['<line x1="300" y1="630" x2="920" y2="630" class="axis"/>']
    for row_index, row in enumerate(ordered.itertuples(index=False)):
        y = 85 + row_index * 52
        lower = 300 + 620 * float(row.pooled_bcr_ci_lower) / max_bcr
        center = 300 + 620 * float(row.pooled_bcr_mean) / max_bcr
        upper = 300 + 620 * float(row.pooled_bcr_ci_upper) / max_bcr
        interval_lines.append(f'<text x="290" y="{y + 5}" text-anchor="end" class="label">{html.escape(row.pair)}</text>')
        interval_lines.append(f'<line x1="{lower:.2f}" y1="{y}" x2="{upper:.2f}" y2="{y}" stroke="#2f6f9f" stroke-width="4"/>')
        interval_lines.append(f'<circle cx="{center:.2f}" cy="{y}" r="6" fill="#b42318"/>')
    interval_lines.append(
        f'<text x="610" y="670" text-anchor="middle" class="label">Pooled descriptive BCR interval (axis maximum {max_bcr:.4f})</text>'
    )
    (figures_dir / "bootstrap_bcr_intervals.svg").write_text(
        svg_document("Phase 5 bootstrap BCR distributions", "\n".join(interval_lines)), encoding="utf-8"
    )

    rank_mean = results["per_replicate_pair_metrics"].pivot_table(
        index="pair", columns="model_seed", values="rank", aggfunc="mean"
    ).sort_index()
    heat_lines = []
    cell_width, cell_height = 61, 47
    left, top = 330, 90
    for column_index, seed in enumerate(rank_mean.columns):
        x = left + column_index * cell_width + cell_width / 2
        heat_lines.append(f'<text x="{x:.1f}" y="75" text-anchor="middle" class="small">{seed}</text>')
    for row_index, (pair, values) in enumerate(rank_mean.iterrows()):
        y = top + row_index * cell_height
        heat_lines.append(f'<text x="320" y="{y + 29}" text-anchor="end" class="label">{html.escape(pair)}</text>')
        for column_index, value in enumerate(values.to_numpy(dtype=float)):
            x = left + column_index * cell_width
            intensity = int(round(230 - 150 * (value - 1) / 9))
            color = f'rgb({intensity},{min(245, intensity + 25)},245)'
            heat_lines.append(f'<rect x="{x}" y="{y}" width="59" height="45" fill="{color}"/>')
            heat_lines.append(f'<text x="{x + 29.5}" y="{y + 29}" text-anchor="middle" class="small">{value:.2f}</text>')
    (figures_dir / "bootstrap_mean_rank_heatmap.svg").write_text(
        svg_document("Mean bootstrap BCR rank by model seed", "\n".join(heat_lines), height=620),
        encoding="utf-8",
    )


def write_phase5_results(results, output_dir, crosscheck, validation, create_figures=True):
    """Write the complete Phase 5 evidence package and its internal manifest."""
    output_dir = Path(output_dir)
    figures_dir = output_dir / "figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    _write_json(output_dir / "phase5_protocol.json", results["protocol"])
    _write_dataframe(output_dir / "phase5_original_pair_metrics.csv", results["original_pair_metrics"])
    _write_dataframe(
        output_dir / "phase5_per_replicate_pair_metrics.csv",
        results["per_replicate_pair_metrics"],
    )
    _write_dataframe(output_dir / "phase5_seed_pair_summary.csv", results["seed_pair_summary"])
    _write_dataframe(output_dir / "phase5_pair_pooled_summary.csv", results["pair_pooled_summary"])
    _write_dataframe(output_dir / "phase5_rank_distribution.csv", results["rank_distribution"])
    selection = dict(results["hard_pair_selection"])
    selection["phase4_crosscheck"] = crosscheck
    _write_json(output_dir / "phase5_hard_pair_selection.json", selection)
    execution = {
        **validation,
        "phase4_crosscheck": crosscheck,
        "final_hard_pairs": selection["H"],
        "final_hard_pair_count": selection["K"],
        "final_hard_pairs_frozen": True,
        "test_accessed": False,
        "execution_device": "CPU",
    }
    _write_json(output_dir / "phase5_execution_summary.json", execution)
    if create_figures:
        _write_figures(results, figures_dir)
    selected_lines = "\n".join(f"- `{pair}`" for pair in selection["H"]) or "- None"
    report = (
        "# Phase 5 Bootstrap Stability Report\n\n"
        "## Material Passport\n\n"
        "- Data scope: frozen validation predictions from ten MentalRoBERTa training seeds.\n"
        "- Statistical unit: validation row, resampled within true class separately per model seed.\n"
        f"- Bootstrap: {results['protocol']['bootstrap_replicates_per_seed']} replicates per seed; "
        f"master seed {results['protocol']['master_bootstrap_seed']}.\n"
        "- Verification: artifact integrity, original Phase 4 reproduction, row-count, finite-value, rank, and manifest gates.\n"
        "- Test access: false. Model weights, raw text, GPU, and network were not used.\n"
        "- Limitation: intervals describe validation-sample stability with trained predictions fixed; they do not estimate retraining or test uncertainty.\n\n"
        "## Frozen hard-pair set H\n\n"
        f"{selected_lines}\n\n"
        f"K = {selection['K']} (K_max = {selection['K_max']}).\n\n"
        "A pair required original top-3 presence in at least 7/10 seeds and mean seed-specific bootstrap "
        "top-3 frequency of at least 0.70. The pooled bootstrap intervals are descriptive only.\n"
    )
    (output_dir / "phase5_report.md").write_text(report, encoding="utf-8")
    manifest_rows = write_sha256_manifest(output_dir)
    return {"execution_summary": execution, "manifest_entries": len(manifest_rows)}


def package_phase5_results(output_dir, zip_path):
    return write_deterministic_zip(
        output_dir, zip_path, prefix="phase5_bootstrap_stability_results"
    )


# 5. Synthetic protocol self-tests


In [5]:
# Deterministic stratification and support preservation.
_toy_true = np.array([0, 0, 1, 1, 1, 2])
_sample_a = stratified_bootstrap_indices(_toy_true, [0, 1, 2], np.random.default_rng(7))
_sample_b = stratified_bootstrap_indices(_toy_true, [0, 1, 2], np.random.default_rng(7))
np.testing.assert_array_equal(_sample_a, _sample_b)
assert np.bincount(_toy_true[_sample_a], minlength=3).tolist() == [2, 3, 1]

# Hand-calculated BCR and exact-tie average ranks.
_toy_confusion = np.array([[8, 2, 0], [3, 7, 0], [0, 1, 9]])
_, _toy_bcr = compute_bcr_vector(_toy_confusion, ["A", "B", "C"])
np.testing.assert_allclose(_toy_bcr, [0.25, 0.0, 0.05], rtol=0.0, atol=1e-15)
_toy_ranks = rank_bcr_vector(np.array([0.5, 0.4, 0.3, 0.3, 0.1]))
np.testing.assert_allclose(_toy_ranks, [1.0, 2.0, 3.5, 3.5, 5.0])
assert (_toy_ranks <= 3.0).tolist() == [True, True, False, False, False]

# Standard linear quantile behavior.
assert abs(np.quantile([0.0, 1.0, 2.0, 3.0], 0.025, method="linear") - 0.075) < 1e-15

# Selection cardinalities: zero, fewer than K, exactly K, and more than K.
_selection_original = pd.DataFrame({
    "pair": ["A|B", "A|C", "A|D", "B|C"],
    "top3_count": [10, 8, 7, 7],
    "median_rank": [1.0, 2.0, 2.0, 3.0],
    "mean_bcr": [0.30, 0.20, 0.25, 0.10],
})
for _frequencies, _expected_k in [
    ([0.1, 0.1, 0.1, 0.1], 0),
    ([0.8, 0.1, 0.1, 0.1], 1),
    ([0.8, 0.8, 0.1, 0.1], 2),
    ([0.8, 0.8, 0.8, 0.8], 3),
]:
    _selection_pooled = pd.DataFrame({
        "pair": _selection_original["pair"],
        "mean_seed_top3_frequency": _frequencies,
    })
    assert select_hard_pairs(_selection_original, _selection_pooled)["K"] == _expected_k

print("Embedded Phase 5 statistical self-tests: PASS")


Embedded Phase 5 statistical self-tests: PASS


# 6. Input discovery and integrity gates


In [6]:
archives = discover_archives(INPUT_DIR)
archive_checks = pd.DataFrame([validate_archive(path) for path in archives])
archive_checks["archive_path"] = archive_checks["archive_path"].astype(str)
assert len(archive_checks) == 10

runs = [load_run(path, expected_rows=1490) for path in archives]
collection_validation = validate_run_collection(
    runs,
    expected_seeds=EXPECTED_SEEDS,
    expected_rows=1490,
    expected_train_rows=6954,
)
assert collection_validation["all_hard_validations_pass"] is True
assert collection_validation["test_accessed"] is False

# Verify the frozen Phase 4 result manifest before using its cross-check table.
phase4_manifest = pd.read_csv(PHASE4_RESULTS_DIR / "sha256_manifest.csv")
for row in phase4_manifest.itertuples(index=False):
    artifact = PHASE4_RESULTS_DIR / row.path
    assert artifact.is_file(), artifact
    assert artifact.stat().st_size == row.bytes, artifact
    assert sha256_file(artifact) == row.sha256, artifact

display(archive_checks[["archive_name", "archive_bytes", "archive_sha256"]])
print("Input archive, collection, no-test-access, and Phase 4 manifest gates: PASS")


,archive_name,archive_bytes,archive_sha256
0,mentalroberta_full_seed1024_len512_e91f034d8a7...,439480,4ebc235f48f1b990bfff860f758729510de544b724da83...
1,mentalroberta_full_seed123_len512_d1ad313ee75d...,437881,283b063df338d1ce50ca9dc60d49f027e91ae7cad024fe...
2,mentalroberta_full_seed13_len512_fea2991778f3_...,441549,23ceddd17dfe24eff0f1367824f8a020d83e606da643b2...
3,mentalroberta_full_seed2026_len512_f6630d94239...,435153,4d07884013a0f5af78f724e33e95a4ebe681a7856e4059...
4,mentalroberta_full_seed211_len512_149308113b40...,438534,56a312411d83f0325d2f6ad107f2c873c69133cd426b4f...
5,mentalroberta_full_seed3407_len512_d8596ea710c...,436962,e8d96433959e8113700e980a8d31c03bd4ee1ea16ddb95...
6,mentalroberta_full_seed340_len512_6353292c8230...,429349,fff4222a9a65067fcd8e3a8c0944e63437d9acd55e353c...
7,mentalroberta_full_seed42_len512_be72f916ca15_...,444578,c45644bf88c9c813742de5613d7bd53c04c65d48f1b302...
8,mentalroberta_full_seed512_len512_412e1caa7392...,439735,9db1ec1a67952c3868160ed365b03052839ac936c1d996...
9,mentalroberta_full_seed77_len512_d14375990239_...,440261,6e3d4ae16d9cff4c4a94ba3964b4c16c434dbb4b4a3c90...


Input archive, collection, no-test-access, and Phase 4 manifest gates: PASS


# 7. Original Phase 4 reproduction


In [7]:
_phase4_recomputed, original_pair_metrics = _original_phase4(runs)
phase4_crosscheck = crosscheck_phase4(
    original_pair_metrics,
    PHASE4_PAIR_METRICS_PATH,
    atol=1e-12,
)
expected_candidates = ["Frustrated|Others", "Suicidal|Depressed", "Depressed|Others"]
assert set(_phase4_recomputed["criterion1_candidates"]["candidate_pairs"]) == set(expected_candidates)
print(json.dumps(phase4_crosscheck, indent=2, sort_keys=True))


{
  "absolute_tolerance": 1e-12,
  "checked_identifier_fields": [
    "model_seed",
    "pair",
    "class_a",
    "class_b"
  ],
  "checked_numeric_fields": [
    "bcr",
    "rank"
  ],
  "row_count": 100,
  "status": "PASS"
}


# 8. Phase 5 stratified bootstrap


In [8]:
analysis_started = time.perf_counter()
phase5 = run_phase5(
    runs,
    labels=LABELS,
    replicate_count=BOOTSTRAP_REPLICATES,
    master_seed=MASTER_BOOTSTRAP_SEED,
)
analysis_elapsed_seconds = time.perf_counter() - analysis_started

long_results = phase5["per_replicate_pair_metrics"]
assert len(long_results) == 200000
assert long_results.groupby("model_seed")["bootstrap_replicate"].nunique().eq(2000).all()
assert long_results.groupby("pair").size().eq(20000).all()

# A real-input deterministic spot check independently repeats five replicates.
_repeat_a = bootstrap_seed(runs[0], replicate_count=5, master_seed=MASTER_BOOTSTRAP_SEED)
_repeat_b = bootstrap_seed(runs[0], replicate_count=5, master_seed=MASTER_BOOTSTRAP_SEED)
pd.testing.assert_frame_equal(_repeat_a, _repeat_b)
print(f"Bootstrap computation complete in {analysis_elapsed_seconds:.3f} seconds on CPU.")


Bootstrap computation complete in 4.161 seconds on CPU.


# 9. Stability summaries and final selection


In [9]:
validation = validate_phase5_results(
    phase5,
    seed_count=10,
    replicate_count=BOOTSTRAP_REPLICATES,
)
selection = phase5["hard_pair_selection"]
assert selection["status"] == "FROZEN"
assert selection["K"] <= K_MAX
display(phase5["pair_pooled_summary"])
display(pd.DataFrame(selection["pair_decisions"]))
print("Final frozen H:", selection["H"])


,pair,seed_count,mean_seed_bcr_mean,mean_seed_bcr_sample_sd,mean_seed_top1_frequency,mean_seed_top3_frequency,median_seed_median_rank,pooled_observation_count,pooled_bcr_mean,pooled_bcr_median,pooled_bcr_ci_lower,pooled_bcr_ci_upper
0,Frustrated|Others,10,0.093305,0.011525,0.80780,1.00000,1.0,20000,0.093305,0.092863,0.070388,0.118631
1,Suicidal|Depressed,10,0.078087,0.012093,0.19145,0.99940,2.0,20000,0.078087,0.077726,0.053907,0.103556
2,Depressed|Others,10,0.046421,0.009405,0.00075,0.87570,3.0,20000,0.046421,0.046058,0.025510,0.069518
3,Depressed|Anxious,10,0.026333,0.007184,0.00000,0.06570,5.0,20000,0.026333,0.025603,0.011369,0.044451
4,Anxious|Others,10,0.025674,0.006064,0.00000,0.03220,5.0,20000,0.025674,0.024881,0.013217,0.042423
5,Suicidal|Frustrated,10,0.021357,0.005994,0.00000,0.01580,6.0,20000,0.021357,0.020948,0.009025,0.036480
6,Suicidal|Others,10,0.020906,0.005329,0.00000,0.00665,6.0,20000,0.020906,0.020705,0.010754,0.032479
7,Depressed|Frustrated,10,0.015782,0.005639,0.00000,0.00405,8.0,20000,0.015782,0.015026,0.005415,0.030051
8,Suicidal|Anxious,10,0.010398,0.004068,0.00000,0.00050,9.0,20000,0.010398,0.009804,0.001701,0.021979
9,Anxious|Frustrated,10,0.006345,0.003343,0.00000,0.00000,10.0,20000,0.006345,0.005415,0.000000,0.014127


,mean_bcr,top3_count,pair,median_rank,mean_seed_top3_frequency,criterion1_pass,criterion2_pass,qualifies
0,0.093227,10,Frustrated|Others,1.0,1.00000,True,True,True
1,0.078162,10,Suicidal|Depressed,2.0,0.99940,True,True,True
2,0.046388,10,Depressed|Others,3.0,0.87570,True,True,True
3,0.026351,0,Depressed|Anxious,4.0,0.06570,False,False,False
4,0.025645,0,Anxious|Others,5.0,0.03220,False,False,False
5,0.020884,0,Suicidal|Others,6.0,0.00665,False,False,False
6,0.021361,0,Suicidal|Frustrated,6.5,0.01580,False,False,False
7,0.015763,0,Depressed|Frustrated,8.0,0.00405,False,False,False
8,0.010341,0,Suicidal|Anxious,9.0,0.00050,False,False,False
9,0.006321,0,Anxious|Frustrated,10.0,0.00000,False,False,False


Final frozen H: ['Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others']


# 10. Results, report, and figures


In [10]:
write_state = write_phase5_results(
    phase5,
    OUTPUT_DIR,
    crosscheck=phase4_crosscheck,
    validation=validation,
    create_figures=True,
)
required_outputs = [
    "phase5_protocol.json",
    "phase5_original_pair_metrics.csv",
    "phase5_per_replicate_pair_metrics.csv",
    "phase5_seed_pair_summary.csv",
    "phase5_pair_pooled_summary.csv",
    "phase5_rank_distribution.csv",
    "phase5_hard_pair_selection.json",
    "phase5_execution_summary.json",
    "phase5_report.md",
    "sha256_manifest.csv",
]
for name in required_outputs:
    assert (OUTPUT_DIR / name).is_file(), name
print(f"Wrote {write_state['manifest_entries']} manifest-covered Phase 5 files.")


Wrote 12 manifest-covered Phase 5 files.


# 11. Manifest and deterministic ZIP verification


In [11]:
manifest = pd.read_csv(OUTPUT_DIR / "sha256_manifest.csv")
for row in manifest.itertuples(index=False):
    artifact = OUTPUT_DIR / row.path
    assert artifact.is_file(), artifact
    assert artifact.stat().st_size == row.bytes, artifact
    assert sha256_file(artifact) == row.sha256, artifact

package_phase5_results(OUTPUT_DIR, RESULT_ZIP_PATH)
with tempfile.TemporaryDirectory() as temporary_directory:
    comparison_zip = Path(temporary_directory) / "comparison.zip"
    package_phase5_results(OUTPUT_DIR, comparison_zip)
    assert sha256_file(RESULT_ZIP_PATH) == sha256_file(comparison_zip)
print("Output manifest and deterministic ZIP verification: PASS")


Output manifest and deterministic ZIP verification: PASS


# 12. Final execution status


In [12]:
final_summary = json.loads((OUTPUT_DIR / "phase5_execution_summary.json").read_text(encoding="utf-8"))
assert final_summary["status"] == "PASS"
assert final_summary["seed_count"] == 10
assert final_summary["bootstrap_replicates_per_seed"] == 2000
assert final_summary["replicate_pair_rows"] == 200000
assert final_summary["test_accessed"] is False
assert final_summary["final_hard_pairs_frozen"] is True

print(json.dumps({
    **final_summary,
    "analysis_elapsed_seconds": round(analysis_elapsed_seconds, 3),
    "result_zip": str(RESULT_ZIP_PATH),
    "result_zip_sha256": sha256_file(RESULT_ZIP_PATH),
    "result_zip_bytes": RESULT_ZIP_PATH.stat().st_size,
}, indent=2, sort_keys=True))
print("PHASE 5 BOOTSTRAP STABILITY: PASS")


{
  "analysis_elapsed_seconds": 4.161,
  "bootstrap_replicates_per_seed": 2000,
  "execution_device": "CPU",
  "final_hard_pair_count": 3,
  "final_hard_pairs": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "final_hard_pairs_frozen": true,
  "finite_metrics": true,
  "pair_count": 10,
  "pair_pooled_summary_rows": 10,
  "phase4_crosscheck": {
    "absolute_tolerance": 1e-12,
    "checked_identifier_fields": [
      "model_seed",
      "pair",
      "class_a",
      "class_b"
    ],
    "checked_numeric_fields": [
      "bcr",
      "rank"
    ],
    "row_count": 100,
    "status": "PASS"
  },
  "rank_flags_consistent": true,
  "replicate_pair_rows": 200000,
  "result_zip": "MentalDistress_Phase5_Bootstrap_Stability_Results.zip",
  "result_zip_bytes": 2263126,
  "result_zip_sha256": "9b8ad4a04ff075ac1eb6a604401c2546a1165f3c038a9ae8eefb6af4f8f6bab5",
  "seed_count": 10,
  "seed_pair_summary_rows": 100,
  "status": "PASS",
  "test_accessed": false
}
PH